# Doğal Dil İşleme + Karar Bilimi (Natural Language Processing + Decision Science)

👩🏻‍🏫 Bu görevde şunları bir araya getireceğiz:
* 🗣 Doğal Dil İşleme (Natural Language Processing)
* 📊 Karar Bilimi (Decision Science)

🎯 Amaç, Olist üzerindeki ürünlerin ve satıcıların **olumsuz (kötü) yorumlarını** anlamaktır.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Data Manipulation
import numpy as np
import pandas as pd
pd.set_option("display.max_columns",None)

# Machine Learning
from sklearn.pipeline import make_pipeline

# Language Processing
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
import string
import unidecode as unidecode

# Vectorizers and NLP Models
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

🕵🏻‍♂️ Olist’in CEO’su [Tiago Dalvi](https://www.linkedin.com/in/tiagodalvi/)’nin senden yorumları okuyup anlamanı istediğini hayal et.

- Müşteriler siparişlerini **1**, **2** veya **3** puanla değerlendirdiklerinde ne söylediler?
- En sık karşılaşılan olumsuz yorumlar neler?
    - En kötü puanlanan ürünler hakkında?
    - En kötü puanlanan satıcılar hakkında?


## (0) Kurulum 🔨

Öncelikle, Olist incelemeleriyle ilgili tüm bilgileri içeren DataFrame'i yükleyeceğiz!

In [3]:
df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/olist_reviews.csv")

In [4]:
df.head()

,review_id,length_review,review_score,order_id,product_category_name,full_review
0,e64fb393e7b32834bb789ff8bb30750e,37,5,658677c97b385a9be170737859d3511b,ferramentas_jardim,Recebi bem antes do prazo estipulado.
1,f7c4243c7fe1938f181bec41a392bdeb,100,5,8e6bfb81e283fa7e4f11123a3fb894f1,esporte_lazer,Parabéns lojas lannister adorei comprar pela ...
2,8670d52e15e00043ae7de4c01cc2fe06,174,4,b9bf720beb4ab3728760088589c62129,eletroportateis,recomendo aparelho eficiente. no site a marca ...
3,4b49719c8a200003f700d3d986ea1a19,45,4,9d6f15f95d01e79bd1349cc208361f09,beleza_saude,"Mas um pouco ,travando...pelo valor ta Boa.\r\n"
4,3948b09f7c818e2d86c9a546758b2335,56,5,e51478e7e277a83743b6f9991dbfa3fb,informatica_acessorios,"Super recomendo Vendedor confiável, produto ok..."


## (2) Metin Temizleme (Text Cleaning)

🧹 `cleaning(sentence)` işlevini oluşturun ve yorumlara uygulayın. **NLTK'da Portekizce lemmatizer bulunmadığını unutmayın** (genellikle bulunmaz, ancak `nltk.stem.RSLPStemmer` kök bulucu vardır).

In [5]:
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer

nltk.download("stopwords")
nltk.download("rslp")

portuguese_stopwords = set(stopwords.words("portuguese"))

# Olumsuzluk bildiren kelimeleri koruyalım.
portuguese_stopwords -= {"não", "nao", "nem", "sem"}

stemmer = RSLPStemmer()

def cleaning(sentence):
    if pd.isna(sentence):
        return ""

    sentence = str(sentence).lower()
    sentence = re.sub(r"\d+", " ", sentence)
    sentence = sentence.translate(
        str.maketrans("", "", string.punctuation)
    )

    words = sentence.split()

    words = [
        stemmer.stem(word)
        for word in words
        if word not in portuguese_stopwords and len(word) > 1
    ]

    return " ".join(words)

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/burak_gokce/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package rslp to /home/burak_gokce/nltk_data...
[nltk_data]   Unzipping stemmers/rslp.zip.


In [6]:
df["full_review_cleaned"] = df["full_review"].apply(cleaning)

df[["full_review", "full_review_cleaned"]].head()

,full_review,full_review_cleaned
0,Recebi bem antes do prazo estipulado.,receb bem ant praz estipul
1,Parabéns lojas lannister adorei comprar pela ...,parabém loj lannist ador compr internet segur ...
2,recomendo aparelho eficiente. no site a marca ...,recom aparelh efici sit marc aparelh impress d...
3,"Mas um pouco ,travando...pelo valor ta Boa.\r\n",pouc travandopel val ta boa
4,"Super recomendo Vendedor confiável, produto ok...",sup recom vend confi produt ok entreg ant praz


## (3) Kötü yorumların analizi

### (3.1) Düşük inceleme puanlarına sahip veri kümesi

😱 1 ile 3 arasında puan alan yorumların oranı nedir? 

In [7]:
bad_review_ratio = df["review_score"].between(1, 3).mean() * 100

print(f"Düşük puanlı yorumların oranı: %{bad_review_ratio:.2f}")

Düşük puanlı yorumların oranı: %26.72


🕵🏻‍♂️ Bu yorumlara odaklanalım...

In [8]:
bad_reviews = df[df["review_score"].between(1, 3)].copy()

print("Düşük puanlı yorum sayısı:", len(bad_reviews))
print("Veri setinin boyutu:", bad_reviews.shape)

bad_reviews[
    ["review_score", "full_review", "full_review_cleaned"]
].head()

Düşük puanlı yorum sayısı: 9658
Veri setinin boyutu: (9658, 7)


,review_score,full_review,full_review_cleaned
14,1,recebi somente 1 controle Midea Split ESTILO....,receb soment control mide split estil falt con...
23,3,Eu comprei duas unidades e só recebi uma e ag...,compr dua unidad receb agor faç
24,3,"Produto bom, porém o que veio para mim não co...",produt bom porém vei mim não condiz fot anúnci
25,1,"Produto muito inferior, mal acabado.",produt inferi mal acab
28,3,Entrega no prazo,entreg praz


### (3.2) Vektörleştirme

🔡 ➡️ 🔢 Metinlerini vektörleştir.

- **Bigram**’leri (iki kelimelik ifadeler) mutlaka hesaba kat.
- Çok sık geçen kelimeleri çıkarmak için `max_df = 0.75` ayarla.
- Spoiler uyarısı: Sonunda **20.000+** kelimeye ulaşacaksın…  
  Bu challenge için sadece `max_features = 5000` ile sınırla.

In [9]:
vectorizer = CountVectorizer(
    ngram_range=(1, 2),
    max_df=0.75,
    max_features=5000
)

X_bow = vectorizer.fit_transform(
    bad_reviews["full_review_cleaned"]
)

print("BoW matris boyutu:", X_bow.shape)
print("Özellik sayısı:", len(vectorizer.get_feature_names_out()))

BoW matris boyutu: (9658, 5000)
Özellik sayısı: 5000


### (3.3) LDA

🕵🏻‍♂️ LDA'yı uyarlayın:
- `n_components = 3` seçin
- *.transform()* ile Konuların Belge Karışımını gösterin
- *.components_* ile Konu Karışımını gösterin

In [10]:
from sklearn.decomposition import LatentDirichletAllocation

lda_model = LatentDirichletAllocation(
    n_components=3,
    random_state=42
)

lda_model.fit(X_bow)

print("Keşfedilen konu sayısı:", lda_model.n_components)
print("Konu-kelime matrisi:", lda_model.components_.shape)

Keşfedilen konu sayısı: 3
Konu-kelime matrisi: (3, 5000)


#### Belge Karışımı (Konular için)

In [11]:
topic_distribution = lda_model.transform(X_bow)

print("Belge-konu matrisi:", topic_distribution.shape)

topic_distribution[:5]

Belge-konu matrisi: (9658, 3)


array([[0.25957585, 0.71573529, 0.02468886],
       [0.94251379, 0.02919127, 0.02829494],
       [0.02088346, 0.95537511, 0.02374142],
       [0.04831903, 0.47714051, 0.47454046],
       [0.83257833, 0.084076  , 0.08334567]])

👉 Her inceleme için en önemli konuyu rapor edelim

In [12]:
bad_reviews["most_important_topic"] = topic_distribution.argmax(axis=1)

bad_reviews[
    ["review_score", "full_review_cleaned", "most_important_topic"]
].head()

,review_score,full_review_cleaned,most_important_topic
14,1,receb soment control mide split estil falt con...,1
23,3,compr dua unidad receb agor faç,0
24,3,produt bom porém vei mim não condiz fot anúnci,1
25,1,produt inferi mal acab,1
28,3,entreg praz,0


#### Konu Karışımı (Kelimeler için)

In [13]:
topic_word_mixture = lda_model.components_

print("Konu-kelime matrisi:", topic_word_mixture.shape)

topic_word_mixture

Konu-kelime matrisi: (3, 5000)


array([[ 5.28990685,  4.5689189 ,  1.39640269, ..., 12.02337263,
        10.31271822,  0.33639193],
       [ 0.34617982,  0.3945179 ,  5.33795354, ...,  0.35203527,
         0.34468889,  0.34369397],
       [ 0.36391333,  8.0365632 , 45.26564377, ...,  4.6245921 ,
         0.34259289,  7.31991411]])

#### Konular

🎁 Size bazı yardımcı fonksiyonlar sağladık:
- `topic_word`: Tek bir konu (topic) için en önemli kelimeleri ve ağırlıklarını döndürür
- `print_topics`: LDA tarafından bulunan farklı konuları, en önemli kelimeleriyle birlikte yazdırır

In [14]:
def topic_word(vectorizer, model, topic, topwords, with_weights = True):
    topwords_indexes = topic.argsort()[:-topwords - 1:-1]
    if with_weights == True:
        topwords = [(vectorizer.get_feature_names_out()[i], round(topic[i],2)) for i in topwords_indexes]
    if with_weights == False:
        topwords = [vectorizer.get_feature_names_out()[i] for i in topwords_indexes]
    return topwords

In [15]:
def print_topics(vectorizer, model, topwords):
    for idx, topic in enumerate(model.components_):
        print("-"*20)
        print("Topic %d:" % (idx))
        print(topic_word(vectorizer, model, topic, topwords))


🕵🏻‍♂️ Konuları en çok kullanılan kelimelerle birlikte yazdırın:

In [16]:
print_topics(vectorizer, lda_model, 10)

--------------------
Topic 0:
[('entreg', 2376.28), ('receb', 2154.13), ('não', 2116.86), ('produt', 1824.47), ('compr', 1421.92), ('cheg', 837.23), ('praz', 726.27), ('ped', 682.44), ('não receb', 677.29), ('falt', 652.37)]
--------------------
Topic 1:
[('produt', 2956.63), ('vei', 740.43), ('não', 729.67), ('nao', 673.9), ('compr', 637.88), ('bom', 626.26), ('receb', 571.64), ('err', 443.91), ('troc', 404.3), ('difer', 384.13)]
--------------------
Topic 2:
[('não', 2224.47), ('produt', 1023.9), ('vei', 907.28), ('compr', 756.2), ('gost', 397.02), ('cor', 374.56), ('recom', 303.5), ('qual', 285.89), ('sem', 270.1), ('cap', 238.63)]


🇧🇷 Burada biraz Brezilya Portekizcesi kelimeler var:
- _cadeiras = chairs_
- _produto = product_
- _recomendo = recommend (não recomendo == not recommend)_
- _bom = good_
- _comprei = bought_
- _veio = came_
- _errado = wrong_
- _gostaria = I would like to..._
- _duas = two_
- _nao = not_
- _entregue = delivered_
- _pecas = part_
- _ainda = yet_
- _recebi = received_

👉 Bir konuyla ilişkili en popüler kelimeleri göster

In [17]:
topic_word_mixture = {
    i: topic_word(
        vectorizer,
        lda_model,
        topic,
        10,
        with_weights=False
    )
    for i, topic in enumerate(lda_model.components_)
}

topic_word_mixture

{0: ['entreg',
  'receb',
  'não',
  'produt',
  'compr',
  'cheg',
  'praz',
  'ped',
  'não receb',
  'falt'],
 1: ['produt',
  'vei',
  'não',
  'nao',
  'compr',
  'bom',
  'receb',
  'err',
  'troc',
  'difer'],
 2: ['não',
  'produt',
  'vei',
  'compr',
  'gost',
  'cor',
  'recom',
  'qual',
  'sem',
  'cap']}

In [20]:
bad_reviews["most_important_words"] = bad_reviews["most_important_topic"].apply(
    lambda i: topic_word_mixture[i]
)

In [22]:
bad_reviews[[
    "review_id",
    "review_score",
    "product_category_name",
    "full_review_cleaned",
    "most_important_topic",
    "most_important_words"
]].head()

,review_id,review_score,product_category_name,full_review_cleaned,most_important_topic,most_important_words
14,e233e51d11511bf30e568c76360ace52,1,eletronicos,receb soment control mide split estil falt con...,1,"[produt, vei, não, nao, compr, bom, receb, err..."
23,8b230a1373c6dc4bd867099fda1d7039,3,ferramentas_jardim,compr dua unidad receb agor faç,0,"[entreg, receb, não, produt, compr, cheg, praz..."
24,cb2fc3e5711b5ae85e0491ee18af63ed,3,beleza_saude,produt bom porém vei mim não condiz fot anúnci,1,"[produt, vei, não, nao, compr, bom, receb, err..."
25,60c714ed14cef913944a3147094a4742,1,moveis_decoracao,produt inferi mal acab,1,"[produt, vei, não, nao, compr, bom, receb, err..."
28,0bd4dcc4f6c4621baf37f73495cad8c4,3,esporte_lazer,entreg praz,0,"[entreg, receb, não, produt, compr, cheg, praz..."


## (3.4) Pipeline Tf-Idf ve LDA

In [23]:
from sklearn import set_config
set_config("diagram")

🔨 Önceki Vectorizer ve LDA'yı birbirine bağlayan bir Pipeline oluşturun.

Temizlenmiş metinlere uyarlayın.

In [24]:
pipeline = make_pipeline(
    TfidfVectorizer(
        ngram_range=(1, 2),
        max_df=0.75,
        max_features=5000
    ),
    LatentDirichletAllocation(
        n_components=3,
        random_state=42
    )
)

pipeline.fit(bad_reviews["full_review_cleaned"])

Pipeline(steps=[('tfidfvectorizer',
                 TfidfVectorizer(max_df=0.75, max_features=5000,
                                 ngram_range=(1, 2))),
                ('latentdirichletallocation',
                 LatentDirichletAllocation(n_components=3, random_state=42))])

💡 `pipeline.components_` ile bileşenlere erişmeye çalışırsanız, Pipeline'da `components_` olmadığı için bu YÜRÜMEZ. Ancak, LDA'ya erişmek için `pipeline._final_estimator` kullanabilirsiniz. Ve buradan konulara erişebilirsiniz!

In [25]:
pipeline._final_estimator

LatentDirichletAllocation(n_components=3, random_state=42)

In [26]:
pipeline._final_estimator.components_

array([[ 0.6679388 ,  1.06949574,  1.53502502, ...,  5.39232082,
         3.06145128,  0.34001862],
       [ 2.17712827,  0.35577808,  0.36387993, ...,  0.35832374,
         0.34243276,  0.34182192],
       [ 0.39230399,  3.51631359, 11.92537559, ...,  0.39046719,
         0.34589248,  3.48816019]])

Pipeline ile **Belge Karışımı**:

In [27]:
pipeline_topic_distribution = pipeline.transform(
    bad_reviews["full_review_cleaned"]
)

print("Belge-konu matrisi:", pipeline_topic_distribution.shape)

pipeline_topic_distribution[:5]

Belge-konu matrisi: (9658, 3)


array([[0.1997441 , 0.72156403, 0.07869187],
       [0.8353526 , 0.08220007, 0.08244733],
       [0.07759609, 0.8382208 , 0.08418311],
       [0.10135084, 0.43566343, 0.46298572],
       [0.74655915, 0.12699155, 0.12644929]])

Pipeline ile **Konu Karışımı**:

In [28]:
pipeline_topic_word = pipeline.named_steps[
    "latentdirichletallocation"
].components_

print("Konu-kelime matrisi:", pipeline_topic_word.shape)

pipeline_topic_word

Konu-kelime matrisi: (3, 5000)


array([[ 0.6679388 ,  1.06949574,  1.53502502, ...,  5.39232082,
         3.06145128,  0.34001862],
       [ 2.17712827,  0.35577808,  0.36387993, ...,  0.35832374,
         0.34243276,  0.34182192],
       [ 0.39230399,  3.51631359, 11.92537559, ...,  0.39046719,
         0.34589248,  3.48816019]])

## (4) 🎁 Ürün Kategorileri

### (4.1) Ürün kategorilerine göre gruplandırma

📈 Veri kümesini `product_category_name` ile gruplandırın ve performanslarını inceleyin.

In [29]:
# Performansa göre ürün kategorileri - sayı, ortalama, medyan ve standart sapmaya bakın
product_categories = df.groupby(by = 'product_category_name').agg({
        'review_score': ["count", "mean", "median", "std"]
    })

# Analiz için belirli bir süreden daha az satılan ürünleri kaldırma
cutoff = 50
product_categories = product_categories[product_categories[("review_score", "count")] > cutoff]

# Ürün kategorilerini performansa göre sıralama
product_categories = product_categories.sort_values(by = [('review_score', 'mean'),
                                                          ('review_score', 'std')],
                                                    ascending = [False, True])
product_categories

review_score                   \
                                                      count      mean median   
product_category_name                                                          
alimentos_bebidas                                        75  4.560000    5.0   
livros_interesse_geral                                  152  4.539474    5.0   
malas_acessorios                                        375  4.317333    5.0   
livros_tecnicos                                          83  4.301205    5.0   
pcs                                                      67  4.283582    5.0   
alimentos                                               151  4.231788    5.0   
papelaria                                               758  4.201847    5.0   
instrumentos_musicais                                   228  4.192982    5.0   
fashion_calcados                                        101  4.188119    5.0   
fashion_bolsas_e_acessorios                             726  4.168044    5.0   
perfumaria                                             1094  4.167276    5.0   
construcao_ferramentas_jardim                            85  4.164706    5.0   
cool_stuff                                             1360  4.154412    5.0   
ferramentas_jardim                                     1391  4.105679    5.0   
beleza_saude                                           3046  4.104071    5.0   
brinquedos                                             1271  4.101495    5.0   
esporte_lazer                                          2637  4.097838    5.0   
bebidas                                                 111  4.090090    5.0   
pet_shop                                                603  4.086235    5.0   
construcao_ferramentas_iluminacao                        86  4.081395    5.0   
eletroportateis                                         242  4.061983    5.0   
automotivo                                             1533  4.049576    5.0   
eletrodomesticos                                        354  4.033898    5.0   
eletrodomesticos_2                                       89  4.011236    5.0   
relogios_presentes                                     2211  4.009950    5.0   
construcao_ferramentas_construcao                       277  4.007220    5.0   
consoles_games                                          339  4.005900    5.0   
utilidades_domesticas                                  2192  4.005018    5.0   
eletronicos                                             942  3.979830    5.0   
industria_comercio_e_negocios                            82  3.963415    5.0   
bebes                                                   882  3.955782    5.0   
sinalizacao_e_seguranca                                  61  3.934426    5.0   
telefonia                                              1654  3.874244    5.0   
cama_mesa_banho                                        3891  3.853765    5.0   
artes                                                    80  3.850000    5.0   
casa_conforto                                           175  3.817143    5.0   
moveis_decoracao                                       2311  3.815232    5.0   
moveis_sala                                             138  3.789855    5.0   
informatica_acessorios                                 2473  3.774767    5.0   
audio                                                   134  3.768657    5.0   
market_place                                            102  3.745098    4.0   
moveis_cozinha_area_de_servico_jantar_e_jardim           94  3.734043    4.0   
construcao_ferramentas_seguranca                         61  3.721311    5.0   
agro_industria_e_comercio                                57  3.701754    4.0   
telefonia_fixa                                           96  3.697917    4.0   
casa_construcao                                         177  3.672316    4.0   
climatizacao                                             93  3.655914    4.0   
fashion_roupa_masculina                        

### (4.2) En kötü ürün kategorileri

👎 *Ortalama değerlendirme puanı* açısından en kötü beş kategoriyi `worst_products` adlı bir değişkene kaydedin.

In [30]:
worst_products = product_categories.tail(5).sort_values(by = [("review_score", "count")],
                                                       ascending = False)
worst_products

review_score                           
                               count      mean median       std
product_category_name                                          
moveis_escritorio                546  3.298535    4.0  1.562366
casa_construcao                  177  3.672316    4.0  1.498094
telefonia_fixa                    96  3.697917    4.0  1.576938
climatizacao                      93  3.655914    4.0  1.625173
fashion_roupa_masculina           56  3.482143    4.0  1.737198

👇 Yalnızca `worst_products` öğelerini içeren bir `worst_products_review` DataFrame oluşturun.

In [31]:
worst_products_reviews = bad_reviews[
    bad_reviews["product_category_name"].isin(worst_products.index)
].copy()

worst_products_reviews[
    [
        "review_id",
        "review_score",
        "product_category_name",
        "full_review_cleaned",
        "most_important_topic",
        "most_important_words"
    ]
].head()

,review_id,review_score,product_category_name,full_review_cleaned,most_important_topic,most_important_words
45,c422274e50e900b46fee429016c5458d,3,moveis_escritorio,gost,2,"[não, produt, vei, compr, gost, cor, recom, qu..."
212,1f836d160f50247a394034abd93b7c24,3,casa_construcao,esper instal,2,"[não, produt, vei, compr, gost, cor, recom, qu..."
253,8f16e30ff188fe57a427fee7afb60c90,1,moveis_escritorio,tamp mes cheg tod amass bat,2,"[não, produt, vei, compr, gost, cor, recom, qu..."
294,70d2258489cae12a65d36c8c09fd54c1,1,moveis_escritorio,compr ros braç branc entreg tod pret,2,"[não, produt, vei, compr, gost, cor, recom, qu..."
306,3171979b1a7f31c323c2807e85246b47,3,moveis_escritorio,compr cade receb praz porém cade vei quebr par...,2,"[não, produt, vei, compr, gost, cor, recom, qu..."


### (4.3). En kötü ürünler için konular

❓ En kötü ürünlerin konuları nelerdir? ❓

In [32]:
worst_products_reviews["most_important_topic"].value_counts()

most_important_topic
0    194
2    129
1     91
Name: count, dtype: int64

In [33]:
bad_frequency = list(worst_products_reviews["most_important_topic"].value_counts().index)
bad_frequency

[0, 2, 1]

In [34]:
[topic_word_mixture[i] for i in bad_frequency]

[['entreg',
  'receb',
  'não',
  'produt',
  'compr',
  'cheg',
  'praz',
  'ped',
  'não receb',
  'falt'],
 ['não',
  'produt',
  'vei',
  'compr',
  'gost',
  'cor',
  'recom',
  'qual',
  'sem',
  'cap'],
 ['produt',
  'vei',
  'não',
  'nao',
  'compr',
  'bom',
  'receb',
  'err',
  'troc',
  'difer']]

## (5) 🎁 Satıcılar...

* En kötü satıcılar tarafından ne tür ürünler satıldı?
* En kötü satıcılar için başlıca yorumlar nelerdir?

### (5.1) En kötü satıcılar

In [35]:
from olist.seller import Seller
sellers = Seller().get_training_data()
sellers.columns

Index(['seller_id', 'seller_city', 'seller_state', 'delay_to_carrier',
       'wait_time', 'date_first_sale', 'date_last_sale', 'months_on_olist',
       'n_orders', 'quantity', 'quantity_per_order', 'sales',
       'share_of_five_stars', 'share_of_one_stars', 'review_score'],
      dtype='object')

👇 En kötü satan 10 ürünü seçin ve bunları `worst_sellers` adlı bir değişkene kaydedin.

In [37]:
worst_sellers = sellers[
    ["seller_id", "review_score", "sales", "n_orders"]
].sort_values(
    by="review_score",
    ascending=True
).head(10)

worst_sellers

,seller_id,review_score,sales,n_orders
2014,1fa2d3def6adfa70e58c276bb64fe5bb,1.0,6.9,1
1444,c87abc38c8ed3240861729e1aeadf221,1.0,133.0,1
2224,f524ad65d7e0f1daab730ef2d2e86196,1.0,578.0,1
864,1f7dfad2cb384ea4d2d7e1ffbd78c407,1.0,49.0,1
2254,c97aa4ee7420f937da13b7f9e2228b99,1.0,29.9,1
1396,9bf11dfc0bec77e5a23028043c3c5a8f,1.0,115.0,1
241,781a66c0d89e30814d49fe8b5e14cd07,1.0,115.0,1
1375,da2782c804606d2a5d8e1760dbb3e7ec,1.0,89.0,1
2215,fc6295add6f51a0936407ead70c1001d,1.0,160.0,1
2320,010da0602d7774602cd1b3f5fb7b709e,1.0,849.5,1


### (5.2) En kötü satıcılar tarafından satılan ürünler

In [43]:
products = data["products"][["product_id", "product_category_name"]].copy()

products = products.rename(
    columns={"product_category_name": "category"}
)

products.head()

,product_id,category
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria
1,3aa071139cb16b67ca9e5dea641aaa2f,artes
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer
3,cef67bcfe19066a932b7673e239eb23d,bebes
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas


In [40]:
from olist.order import Order

order = Order()

result = order.get_wait_time()

print("Dönen veri tipi:", type(result))
print("Sonuç:", result)

Dönen veri tipi: <class 'NoneType'>
Sonuç: None


In [42]:
from olist.data import Olist

data = Olist().get_data()

print(data.keys())
print(data["products"].columns.tolist())

dict_keys(['sellers', 'customers', 'product_category_name_translation', 'order_payments', 'order_items', 'products', 'geolocation', 'order_reviews', 'orders'])
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


❓ En kötü satıcılar tarafından satılan ürün türleri nelerdir? ❓

In [44]:
sellers_product_category = data["order_items"].merge(products,
                                             on = "product_id", how = "left")[["seller_id", "category"]]

sellers_product_category

,seller_id,category
0,48436dade18ac8b2bce089ec2a041202,cool_stuff
1,dd7ddc04e1b6c2c614352b383efe2d36,pet_shop
2,5b51032eddd242adc84c38acab88f23d,moveis_decoracao
3,9d7a1d34a5052409006425275ba1c2b4,perfumaria
4,df560393f3a51e74553ab94004ba5c87,ferramentas_jardim
...,...,...
112645,b8bc237ba3788b23da09c0f1f3a3288c,utilidades_domesticas
112646,f3c38ab652836d21de61fb8314b69182,informatica_acessorios
112647,c3cfdc648177fdbbbb35635a37472c53,esporte_lazer
112648,2b3e4a2a3ea8e01938cabda2a3e5cc79,informatica_acessorios


In [45]:
sellers_product_category.groupby("seller_id").count()

,category
seller_id,
0015a82c2db000af6aaaf3ae2ecb0532,3
001cca7ae9ae17fb1caed9dfb1094831,239
001e6ad469a905060d959994f1b41e4f,1
002100f778ceb8431b7a1020ff7ab48f,55
003554e2dce176b5555353e4f3555ac8,0
...,...
ffcfefa19b08742c5d315f2791395ee5,1
ffdd9f82b9a447f6f8d4b91554cc7dd3,20
ffeee66ac5d5a62fe688b9d26f83f534,14


### (5.3) En kötü satıcılar için kategoriler ve konular

🎁 İşte bazı kullanışlı işlevler:
- Bir satıcı tarafından satılan ürün kategorilerini göstermek için `focus_seller(seller_id)`
- Bir satıcı için en sık kullanılan konuların en popüler kelimelerini göstermek için `bad_reviews_seller`

In [46]:
def focus_seller(seller_id):
    return sellers_product_category[sellers_product_category.seller_id == seller_id].value_counts()

In [47]:
print("Yorum tablosu:", worst_products_reviews.columns.tolist())
print("Sipariş tablosu:", data["order_items"].columns.tolist())

Yorum tablosu: ['review_id', 'length_review', 'review_score', 'order_id', 'product_category_name', 'full_review', 'full_review_cleaned', 'most_important_topic', 'most_important_words']
Sipariş tablosu: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']


In [52]:
bad_reviews_sellers = bad_reviews.merge(
    data["order_items"],
    on="order_id",
    how="inner"
)

bad_reviews_sellers.head(3)

,review_id,length_review,review_score,order_id,product_category_name,full_review,full_review_cleaned,most_important_topic,most_important_words,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,e233e51d11511bf30e568c76360ace52,97,1,548df2c6e5f089574614894bca78acf5,eletronicos,recebi somente 1 controle Midea Split ESTILO....,receb soment control mide split estil falt con...,1,"[produt, vei, não, nao, compr, bom, receb, err...",1,62ad9a8972411e333e16347051a98e2a,128639473a139ac0f3e5f5ade55873a5,2018-01-16 16:49:36,28.90,3.94
1,e233e51d11511bf30e568c76360ace52,97,1,548df2c6e5f089574614894bca78acf5,eletronicos,recebi somente 1 controle Midea Split ESTILO....,receb soment control mide split estil falt con...,1,"[produt, vei, não, nao, compr, bom, receb, err...",2,566a4f2c4385f36d15c00dfcaae132d1,8b321bb669392f5163d04c59e235e066,2018-01-16 16:49:36,21.90,26.26
2,8b230a1373c6dc4bd867099fda1d7039,60,3,071251fe3b3493294536f03737a8a679,ferramentas_jardim,Eu comprei duas unidades e só recebi uma e ag...,compr dua unidad receb agor faç,0,"[entreg, receb, não, produt, compr, cheg, praz...",1,d2085f7e0f9533605386960fc7e987ec,aac29b1b99776be73c3049939652091d,2017-11-06 02:56:07,12.49,15.10


In [49]:
def bad_reviews_seller(bad_reviews_sellers, seller_id):
    mask = (bad_reviews_sellers.seller_id == seller_id)
    temp = bad_reviews_sellers[mask]
    if len(temp) > 0: # satıcı kötü yorumlar veri çerçevesinde görünüyorsa
        most_frequent_topic_seller = list(temp.most_important_topic.value_counts().head(1).index)[0]
        return topic_word_mixture[most_frequent_topic_seller]

❓Bu en az satan ürünlerin her biri için en sık kullanılan ürün kategorilerini ve kelimeleri gösterin ❓

In [50]:
for worst_seller in worst_sellers["seller_id"]:
    print("-"*50)
    print(f"Focusing on the seller #{worst_seller}...")
    print(focus_seller(worst_seller))
    print(bad_reviews_seller(bad_reviews_sellers, worst_seller))


--------------------------------------------------
Focusing on the seller #1fa2d3def6adfa70e58c276bb64fe5bb...
seller_id                         category             
1fa2d3def6adfa70e58c276bb64fe5bb  utilidades_domesticas    1
Name: count, dtype: int64
None
--------------------------------------------------
Focusing on the seller #c87abc38c8ed3240861729e1aeadf221...
seller_id                         category                   
c87abc38c8ed3240861729e1aeadf221  fashion_bolsas_e_acessorios    1
Name: count, dtype: int64
None
--------------------------------------------------
Focusing on the seller #f524ad65d7e0f1daab730ef2d2e86196...
seller_id                         category          
f524ad65d7e0f1daab730ef2d2e86196  ferramentas_jardim    2
Name: count, dtype: int64
None
--------------------------------------------------
Focusing on the seller #1f7dfad2cb384ea4d2d7e1ffbd78c407...
seller_id                         category
1f7dfad2cb384ea4d2d7e1ffbd78c407  bebes       1
Name: count, dt

In [51]:
seller_ids = worst_sellers["seller_id"]

matches = bad_reviews_sellers[
    bad_reviews_sellers["seller_id"].isin(seller_ids)
]

print("Eşleşen yorum sayısı:", len(matches))
print("Eşleşen satıcı sayısı:", matches["seller_id"].nunique())

Eşleşen yorum sayısı: 0
Eşleşen satıcı sayısı: 0


In [53]:
seller_ids = worst_sellers["seller_id"]

matches = bad_reviews_sellers[
    bad_reviews_sellers["seller_id"].isin(seller_ids)
]

print("Eşleşen yorum satırı sayısı:", len(matches))
print("Eşleşen satıcı sayısı:", matches["seller_id"].nunique())

Eşleşen yorum satırı sayısı: 12
Eşleşen satıcı sayısı: 5


In [54]:
for worst_seller in worst_sellers["seller_id"]:
    print("-"*50)
    print(f"Focusing on the seller #{worst_seller}...")
    print(focus_seller(worst_seller))
    print(bad_reviews_seller(bad_reviews_sellers, worst_seller))

--------------------------------------------------
Focusing on the seller #1fa2d3def6adfa70e58c276bb64fe5bb...
seller_id                         category             
1fa2d3def6adfa70e58c276bb64fe5bb  utilidades_domesticas    1
Name: count, dtype: int64
['entreg', 'receb', 'não', 'produt', 'compr', 'cheg', 'praz', 'ped', 'não receb', 'falt']
--------------------------------------------------
Focusing on the seller #c87abc38c8ed3240861729e1aeadf221...
seller_id                         category                   
c87abc38c8ed3240861729e1aeadf221  fashion_bolsas_e_acessorios    1
Name: count, dtype: int64
['não', 'produt', 'vei', 'compr', 'gost', 'cor', 'recom', 'qual', 'sem', 'cap']
--------------------------------------------------
Focusing on the seller #f524ad65d7e0f1daab730ef2d2e86196...
seller_id                         category          
f524ad65d7e0f1daab730ef2d2e86196  ferramentas_jardim    2
Name: count, dtype: int64
None
--------------------------------------------------
Focusi

🏁 Tebrikler. NLP'nin bazı temellerini (Ön İşleme + Vektörleştirme + NB/LDA) öğrendiniz ve bu yeni “uzmanlığı” Karar Bilimi ile birleştirdik.

💾 `git add / commit / push` yapmayı unutmayın.